# Część 2: Moduł `typing` i zaawansowane adnotacje w praktyce

W tej części przejdziemy od prostych typów (`int`, `str`, `float`) do struktur wykorzystywanych w codziennej pracy z danymi:
1. Adnotacje zmiennych i komentarze typów (*Type Comments* / *Variable Annotations*)
2. Typowanie kolekcji: listy, krotki i słowniki (*Sequences and Mappings*)
3. Aliasy typów (*Type Aliases*)
4. Funkcje nic niezwracające oraz typ `Any` (*Functions Without Return Values* & *The `Any` Type*)
5. Elastyczność i braki danych: `Union` oraz `Optional` (*Example: A Deck of Cards* / *Optional*)

## 1. Adnotacje zmiennych (Variable Annotations vs Type Comments)\

*Źródło w artykule: **Type Comments** / **Playing With Python Types, Part 1***

Adnotacje można dodawać nie tylko do argumentów funkcji, ale również do pojedynczych zmiennych — szczególnie wtedy, gdy inicjalizujemy pustą strukturę lub typ nie wynika wprost z przypisania.

* **Nowoczesna składnia (PEP 526, Python 3.6+):** `zmienna: typ = wartość`
* **Starsza składnia (Type Comments):** `# type: typ` (spotykana w starszym kodzie lub przy wsparciu dla Pythona 2).

In [1]:
# Nowoczesna adnotacja zmiennej
liczba_rekordow: int = 1500
srednia_ocena: float

# Starszy zapis (Type Comment) – traktowany przez interpreter jak zwykły komentarz,
# ale odczytywany przez narzędzia takie jak mypy:
nazwy_kolumn = []  # type: list[str]

# Python przechowuje adnotacje zmiennych na poziomie modułu w słowniku __annotations__
print("Adnotacje w bieżącym zakresie:", __annotations__)

Adnotacje w bieżącym zakresie: {'liczba_rekordow': <class 'int'>, 'srednia_ocena': <class 'float'>}


## 2. Kolekcje: Listy, Krotki i Słowniki
*Źródło w artykule: **Playing With Python Types, Part 1 -> Sequences and Mappings***

Samo oznaczenie `list` lub `dict` mówi tylko, czym jest kontener, ale nie mówi, **jakie dane znajdują się w środku**. W analizie danych kluczowe jest określenie typu elementów wewnątrz kolekcji.

* **Moduł `typing` (Python 3.5+):** `List[str]`, `Dict[str, float]`, `Tuple[int, str]`
* **Wbudowane generyki (PEP 585, Python 3.9+):** `list[str]`, `dict[str, float]`, `tuple[int, str]` *(zalecane w nowoczesnym kodzie)*

In [2]:
from typing import Dict, List, Tuple

# 1. Lista o jednorodnym typie elementów
pomiary_temperatur: List[float] = [21.5, 22.0, 19.8, 23.1]

# 2. Słownik mapujący klucze (str) na wartości (int)
liczba_wystapien: Dict[str, int] = {
    "Warszawa": 120,
    "Gdańsk": 85,
    "Kraków": 95,
}

# 3. Krotka (Tuple) o stałej długości i konkretnych typach na każdej pozycji
rekord_studenta: Tuple[int, str, float] = (1042, "Anna Kowalska", 4.75)

# 4. Krotka o zmiennej długości tego samego typu (używamy wielokropka ...)
wspolrzedne_wielokata: Tuple[float, ...] = (54.35, 18.64, 54.36, 18.65, 54.37, 18.66)

# Nowoczesny odpowiednik w Pythonie 3.9+ (bez importowania z typing):
nowoczesna_lista: list[float] = [1.0, 2.5, 3.8]
nowoczesny_slownik: dict[str, int] = {"A": 1, "B": 2}

print(f"Rekord: {rekord_studenta}, Liczba pomiarów: {len(pomiary_temperatur)}")

Rekord: (1042, 'Anna Kowalska', 4.75), Liczba pomiarów: 4


## 3. Aliasy typów (Type Aliases)
*Źródło w artykule: **Playing With Python Types, Part 1 -> Type Aliases***

Gdy pracujemy ze złożonymi strukturami danych (np. lista krotek reprezentujących punkty na mapie lub wektory cech), powtarzanie długich definicji pogarsza czytelność kodu. **Alias typu** pozwala nadać takiej strukturze własną, czytelną nazwę.

In [3]:
from typing import List, Tuple

# Definiujemy własne aliasy typów (konwencja: CamelCase, jak nazwy klas)
Punkt2D = Tuple[float, float]
SciezkaGPS = List[Punkt2D]

def oblicz_dlugosc_trasy(trasa: SciezkaGPS) -> float:
    """Przyjmuje listę współrzędnych (Punkt2D) i zwraca sumaryczny dystans."""
    dystans = 0.0
    for i in range(1, len(trasa)):
        x1, y1 = trasa[i - 1]
        x2, y2 = trasa[i]
        dystans += ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5
    return round(dystans, 4)

moja_trasa: SciezkaGPS = [(54.35, 18.64), (54.36, 18.65), (54.38, 18.67)]
print("Długość trasy:", oblicz_dlugosc_trasy(moja_trasa))

Długość trasy: 0.0424


## 4. Funkcje nic niezwracające (`-> None`) oraz typ `Any`
*Źródło w artykule: **Playing With Python Types, Part 1 -> Functions Without Return Values** oraz **The `Any` Type***

* **`-> None`:** W Pythonie funkcja bez instrukcji `return` domyślnie zwraca `None`. W adnotacjach oznaczamy to jawnie jako `-> None` (np. przy funkcjach wyświetlających raport lub zapisujących dane do pliku).
* **`Any`:** Specjalny typ oznaczający „dowolny typ”. Wyłącza sprawdzanie typów dla danej zmiennej. Przydaje się przy integracji ze starym kodem lub nieustrukturyzowanymi danymi (np. surowy JSON), ale jego nadużywanie niweczy sens statycznej analizy.

In [4]:
from typing import Any, List

def wyswietl_raport(tytul: str, wartosci: List[Any]) -> None:
    """Funkcja wykonuje efekt uboczny (print) i nic nie zwraca."""
    print(f"=== {tytul} ===")
    for element in wartosci:
        # element ma typ Any, więc możemy przekazać tu dowolny obiekt
        print(f" - {element} (typ: {type(element).__name__})")

# Lista mieszana: tekst, liczba, wartość logiczna
surowe_dane: List[Any] = ["Sensor_01", 42.7, True, None]
wyswietl_raport("Odczyt z czujnika", surowe_dane)

=== Odczyt z czujnika ===
 - Sensor_01 (typ: str)
 - 42.7 (typ: float)
 - True (typ: bool)
 - None (typ: NoneType)


## 5. Wiele typów i obsługa `None`: `Union` oraz `Optional`
*Źródło w artykule: **Playing With Python Types, Part 1 -> Example: A Deck of Cards / Optional***

W przetwarzaniu danych bardzo często spotykamy sytuacje, w których:
1. Zmienna może przyjmować **kilka różnych typów** (np. identyfikator jako `int` lub `str`) $\rightarrow$ `Union[int, str]` (w Pythonie 3.10+ zapisywane skrótowo jako `int | str`).
2. Wartość może **brakować** (np. `None` w niekompletnym rekordzie) $\rightarrow$ `Optional[float]`, co jest dokładnym skrótem dla `Union[float, None]`.

In [5]:
%%writefile analiza_danych_demo.py
from typing import Optional, Union

# Union: funkcja akceptuje zarówno liczbę całkowitą, jak i napis
def normalizuj_id(user_id: Union[int, str]) -> str:
    return str(user_id).strip().zfill(5)

# Optional: wynik może być liczbą zmiennoprzecinkową albo None (gdy lista jest pusta)
def bezpieczna_srednia(liczby: list[float]) -> Optional[float]:
    if not liczby:
        return None
    return sum(liczby) / len(liczby)

print(normalizuj_id(42))
print(normalizuj_id("7"))

wynik = bezpieczna_srednia([])
# BŁĄD DLA MYPY: wynik może być None, więc nie można bezpiecznie dodać do niego liczby!
print(wynik + 10)

Writing analiza_danych_demo.py


In [6]:
!mypy analiza_danych_demo.py

<class 'OSError'>: Not available